# Tuning A12: Đối chứng chỉ dùng lịch

Mục tiêu: tìm tham số riêng cho từng mô hình trên cấu hình A12 cố định.

Kết luận và tham số được chọn lấy từ bảng cuối notebook sau khi chạy. C30 là đường chính; A12/B27 kiểm tra đóng góp của nhóm đặc trưng sau tuning.

Cả ba gọi `src/tuning_cv.py`, dùng cohort và ba fold của FE, cùng danh sách ứng viên và ngân sách. Notebook không chấm validation/test. Các hàm nằm trong module dùng chung; mỗi ô code có giải thích ngay phía trước.

A12 đều dùng 22.503 giờ đủ history, cùng ba fold, chấm đúng 16.875 giờ OOF. Module gọi trực tiếp load_protocol và prepare_fold; không điền median cho lag.

## 1. Nạp module

Tìm gốc dự án từ CSV theo giờ. Mỗi notebook chạy bằng namespace mới để không mang trạng thái của lượt tuning khác.

In [1]:
from pathlib import Path
import sys
project_root = next(p for p in (Path.cwd(), *Path.cwd().parents)
                    if (p / 'data/preprocessed/hourly_unfilled.csv').is_file())
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
import pandas as pd
from IPython.display import Markdown, display
from src import tuning_cv as cv
CONFIG = 'A12'
CONFIGS = (CONFIG,)
features = cv.CONFIGS[CONFIG]


## 2. Dùng cohort và fold của FE

`load_context` gọi `load_protocol` rồi `prepare_fold` trong `src/feature_experiments.py`. Mọi cấu hình đều lọc cùng các giờ có đủ lag_24h, lag_168h và mean_24h, kể cả A12/B27. Không điền median cho lag. `prepare_fold` chỉ học median thời tiết từ đúng fold-train, giữ các quan sát weather gốc trước t làm ngữ cảnh.

Ma trận chung có 30 cột không thiếu; mỗi mô hình chỉ chọn đúng cột của CONFIG. Ridge có thêm `weekday_hour = day_of_week*24 + hour`, rồi one-hot thứ–giờ và tháng trong fold. Đây là đổi biểu diễn lịch, không thêm thông tin ngoài lịch. Các cột số của Ridge và đầu vào MLP được chuẩn hóa chỉ trên fold-train.

In [2]:
context = cv.load_context(project_root)
display(pd.DataFrame([dict(config=CONFIG, n_features=len(features), n=len(context['cohort']),
                           n_train=len(context['train']), n_oof=int(context['manifest'].n.sum()),
                           protocol_hash=context['protocol_hash'])]))
display(cv.describe_folds(context))
display(cv.budget_table(context))
display(Markdown('**Sáu ứng viên RF được lấy trực tiếp từ lưới:**'))
display(pd.DataFrame(context['tuning_protocol']['candidates']['Random Forest']))

,config,n_features,n,n_train,n_oof,protocol_hash
0,A12,12,22503,28402,16875,6042efc8e56db7968fddaa33ee9d8b43d53c817fb778b1...


,fold,fit_start,fit_end,score_start,score_end,n_fit,n,fit_calendar_hours,score_calendar_hours,fit_hash,score_hash
0,1,2012-10-09 09:00:00,2013-08-05 19:00:00,2013-08-05 20:00:00,2015-07-12 06:00:00,5628,5625,7211.0,16931.0,907f339dcf7453df3f5e70f83fd477f72bdfd531e01eae...,3e445aa0bc2af81e32b3d2126a6f9e34073d17a6872bc0...
1,2,2012-10-09 09:00:00,2015-07-12 06:00:00,2015-07-12 07:00:00,2016-09-07 17:00:00,11253,5625,24142.0,10163.0,0a956b96e0a294dfd75ca7d8b49aa36b9e6cbffd755794...,a1d7f7f4f11fc1da3571b8dfdba463987d69e3ef1adee3...
2,3,2012-10-09 09:00:00,2016-09-07 17:00:00,2016-09-07 18:00:00,2017-05-10 04:00:00,16878,5625,34305.0,5867.0,a64f6fc3241daadbcf0a479c22878e11e10c69bfa3c4e2...,4117d3a132e44ecc9c0f8eea01e43dc911e7758ea92dc3...


,model,candidates,folds,fits_per_config,n
0,Ridge,6,3,18,22503
1,Random Forest,6,3,18,22503
2,Gradient Boosting,6,3,18,22503
3,MLP,6,3,18,22503
4,XGBoost,6,3,18,22503


**Sáu ứng viên RF được lấy trực tiếp từ lưới:**

,model__n_estimators,model__min_samples_leaf,model__max_features,model__max_depth
0,100,1,0.80,8
1,100,5,0.33,8
2,100,10,1.00,12
3,200,1,0.50,16
4,400,5,0.33,16
5,200,2,0.80,20


## 3. Chạy CV và hai baseline

Mỗi mô hình dùng sáu ứng viên × ba fold, cùng danh sách giữa A/B/C. Mỗi fold tạo pipeline mới. Điền thiếu đã do FE xử lý nên pipeline không có imputer thứ hai. Chọn MAE trung bình fold, RMSE phá hòa; đồng thời lưu R², SD và cảnh báo hội tụ.

Hai baseline cũng chấm trên đúng các giờ này: trung bình thứ–giờ học từ fold-train và seasonal naive lấy lag_168h. OOF là dự đoán trên các phần score nằm ngoài phần học của từng fold.

Ô sau thực sự huấn luyện lại. Kết quả của riêng CONFIG ghi vào `outputs/CONFIG/results.json`, kèm `protocol.json`; chạy cấu hình này không ghi đè kết quả cấu hình khác. History ở score được cập nhật bằng các quan sát quá khứ đã đến; chưa đại diện dự báo cả đoạn tương lai từ một mốc.

In [3]:
result = cv.run_configuration(context, CONFIG)

A12 | Ridge | candidate 1/6 | fold 1: MAE=288.615
A12 | Ridge | candidate 1/6 | fold 2: MAE=353.658
A12 | Ridge | candidate 1/6 | fold 3: MAE=308.827
A12 | Ridge | candidate 2/6 | fold 1: MAE=288.698
A12 | Ridge | candidate 2/6 | fold 2: MAE=353.677
A12 | Ridge | candidate 2/6 | fold 3: MAE=308.846
A12 | Ridge | candidate 3/6 | fold 1: MAE=289.564
A12 | Ridge | candidate 3/6 | fold 2: MAE=353.866
A12 | Ridge | candidate 3/6 | fold 3: MAE=309.041
A12 | Ridge | candidate 4/6 | fold 1: MAE=302.007
A12 | Ridge | candidate 4/6 | fold 2: MAE=356.517
A12 | Ridge | candidate 4/6 | fold 3: MAE=311.377
A12 | Ridge | candidate 5/6 | fold 1: MAE=525.413
A12 | Ridge | candidate 5/6 | fold 2: MAE=421.556
A12 | Ridge | candidate 5/6 | fold 3: MAE=359.307
A12 | Ridge | candidate 6/6 | fold 1: MAE=1310.741
A12 | Ridge | candidate 6/6 | fold 2: MAE=1051.806
A12 | Ridge | candidate 6/6 | fold 3: MAE=905.873
A12 | Random Forest | candidate 1/6 | fold 1: MAE=266.849
A12 | Random Forest | candidate 1/6 | fo

## 4. Kết quả và tham số được chọn

MAE/RMSE có đơn vị xe/giờ, R² không có đơn vị. SD chỉ mô tả biến động giữa ba giai đoạn, không phải ngưỡng kiểm định thống kê. Các fold bằng số hàng chấm nhưng khác thời lượng lịch. Xem cảnh báo hội tụ trong JSON trước khi nhận xét MLP.

Kết quả chỉ dùng train CV; không tự fit lại hoặc đánh giá validation/test. Ensemble về sau đọc tham số và OOF đã lưu, không tìm lại tham số mô hình thành phần.

In [4]:
display(cv.results_table(result))
for model in result['results']:
    display(Markdown(f"**{model['model']}** — tham số được chọn: `{model['best_params']}`"))
    display(pd.DataFrame(model['folds']))
best = cv.results_table(result).loc[lambda t: ~t.model.isin(cv.BASELINES)].sort_values(['MAE_mean', 'RMSE_mean']).iloc[0]
display(Markdown(f"Trong {CONFIG}, mô hình có MAE CV thấp nhất là **{best['model']}**, MAE {best.MAE_mean:.3f}, RMSE {best.RMSE_mean:.3f} xe/giờ. Chưa phải kết quả holdout."))
rf = next(m for m in result['results'] if m['model'] == 'Random Forest')
display(Markdown('**Kiểm tra sáu ứng viên RF đã thực sự chạy:**'))
display(pd.DataFrame([dict(candidate=t['candidate'], **t['parameters'], MAE_CV=t['MAE_mean'], RMSE_CV=t['RMSE_mean'], n=sum(f['n'] for f in t['folds'])) for t in rf['candidates']]))

,config,model,n,MAE_mean,MAE_std,RMSE_mean,RMSE_std,R2_mean,R2_std
0,A12,Mean weekday-hour,16875,305.628321,9.218800,512.521230,51.706045,0.931526,0.012548
1,A12,Seasonal naive 168h,16875,336.788859,25.656439,619.320594,79.984387,0.899629,0.023644
2,A12,Ridge,16875,317.033621,33.288909,487.929444,58.752916,0.937639,0.015329
3,A12,Random Forest,16875,288.182262,20.674355,463.379493,39.654253,0.944023,0.009874
4,A12,Gradient Boosting,16875,300.498659,34.274467,470.428977,54.829987,0.942034,0.014175
5,A12,MLP,16875,303.579039,39.273389,475.703933,65.207970,0.940525,0.016969
6,A12,XGBoost,16875,290.449713,34.487217,464.275801,47.528842,0.943662,0.012119


**Ridge** — tham số được chọn: `{'model__alpha': 0.001}`

,fold,fit_start,fit_end,score_start,score_end,n_fit,n,fit_calendar_hours,score_calendar_hours,fit_hash,score_hash,MAE,RMSE,R2,seconds,warnings
0,1,2012-10-09 09:00:00,2013-08-05 19:00:00,2013-08-05 20:00:00,2015-07-12 06:00:00,5628,5625,7211.0,16931.0,907f339dcf7453df3f5e70f83fd477f72bdfd531e01eae...,3e445aa0bc2af81e32b3d2126a6f9e34073d17a6872bc0...,288.615495,479.045823,0.941713,0.065251,[]
1,2,2012-10-09 09:00:00,2015-07-12 06:00:00,2015-07-12 07:00:00,2016-09-07 17:00:00,11253,5625,24142.0,10163.0,0a956b96e0a294dfd75ca7d8b49aa36b9e6cbffd755794...,a1d7f7f4f11fc1da3571b8dfdba463987d69e3ef1adee3...,353.658282,550.618281,0.920685,0.026200,[]
2,3,2012-10-09 09:00:00,2016-09-07 17:00:00,2016-09-07 18:00:00,2017-05-10 04:00:00,16878,5625,34305.0,5867.0,a64f6fc3241daadbcf0a479c22878e11e10c69bfa3c4e2...,4117d3a132e44ecc9c0f8eea01e43dc911e7758ea92dc3...,308.827086,434.124229,0.950519,0.031822,[]


**Random Forest** — tham số được chọn: `{'model__n_estimators': 100, 'model__min_samples_leaf': 1, 'model__max_features': 0.8, 'model__max_depth': 8}`

,fold,fit_start,fit_end,score_start,score_end,n_fit,n,fit_calendar_hours,score_calendar_hours,fit_hash,score_hash,MAE,RMSE,R2,seconds,warnings
0,1,2012-10-09 09:00:00,2013-08-05 19:00:00,2013-08-05 20:00:00,2015-07-12 06:00:00,5628,5625,7211.0,16931.0,907f339dcf7453df3f5e70f83fd477f72bdfd531e01eae...,3e445aa0bc2af81e32b3d2126a6f9e34073d17a6872bc0...,266.848695,455.747230,0.947245,0.237520,[]
1,2,2012-10-09 09:00:00,2015-07-12 06:00:00,2015-07-12 07:00:00,2016-09-07 17:00:00,11253,5625,24142.0,10163.0,0a956b96e0a294dfd75ca7d8b49aa36b9e6cbffd755794...,a1d7f7f4f11fc1da3571b8dfdba463987d69e3ef1adee3...,308.127415,506.295128,0.932940,0.334571,[]
2,3,2012-10-09 09:00:00,2016-09-07 17:00:00,2016-09-07 18:00:00,2017-05-10 04:00:00,16878,5625,34305.0,5867.0,a64f6fc3241daadbcf0a479c22878e11e10c69bfa3c4e2...,4117d3a132e44ecc9c0f8eea01e43dc911e7758ea92dc3...,289.570677,428.096121,0.951884,0.456367,[]


**Gradient Boosting** — tham số được chọn: `{'model__n_estimators': 200, 'model__max_depth': 3, 'model__learning_rate': 0.1}`

... (4 outputs left out)

,fold,fit_start,fit_end,score_start,score_end,n_fit,n,fit_calendar_hours,score_calendar_hours,fit_hash,score_hash,MAE,RMSE,R2,seconds,warnings
0,1,2012-10-09 09:00:00,2013-08-05 19:00:00,2013-08-05 20:00:00,2015-07-12 06:00:00,5628,5625,7211.0,16931.0,907f339dcf7453df3f5e70f83fd477f72bdfd531e01eae...,3e445aa0bc2af81e32b3d2126a6f9e34073d17a6872bc0...,255.407376,445.671195,0.949552,0.261944,[]
1,2,2012-10-09 09:00:00,2015-07-12 06:00:00,2015-07-12 07:00:00,2016-09-07 17:00:00,11253,5625,24142.0,10163.0,0a956b96e0a294dfd75ca7d8b49aa36b9e6cbffd755794...,a1d7f7f4f11fc1da3571b8dfdba463987d69e3ef1adee3...,324.353619,518.292669,0.929724,0.366161,[]
2,3,2012-10-09 09:00:00,2016-09-07 17:00:00,2016-09-07 18:00:00,2017-05-10 04:00:00,16878,5625,34305.0,5867.0,a64f6fc3241daadbcf0a479c22878e11e10c69bfa3c4e2...,4117d3a132e44ecc9c0f8eea01e43dc911e7758ea92dc3...,291.588144,428.863539,0.951711,0.449215,[]


Trong A12, mô hình có MAE CV thấp nhất là **Random Forest**, MAE 288.182, RMSE 463.379 xe/giờ. Chưa phải kết quả holdout.

**Kiểm tra sáu ứng viên RF đã thực sự chạy:**

,candidate,model__n_estimators,model__min_samples_leaf,model__max_features,model__max_depth,MAE_CV,RMSE_CV,n
0,1,100,1,0.80,8,288.182262,463.379493,16875
1,2,100,5,0.33,8,297.782126,464.090087,16875
2,3,100,10,1.00,12,289.519129,471.737653,16875
3,4,200,1,0.50,16,296.785625,473.090799,16875
4,5,400,5,0.33,16,288.542641,462.629754,16875
5,6,200,2,0.80,20,295.008407,473.299402,16875


## 5. Bàn giao

Mở `tuning_compare.ipynb` sau khi chạy đủ ba cấu hình. Notebook so sánh chỉ đọc JSON và dừng nếu protocol, danh sách ứng viên hoặc timestamp OOF khác nhau. Tham số tối ưu có thể khác giữa cấu hình và mô hình; không dùng một bộ tham số chung sau khi tuning.